# 02 — Text Cleaning & Section Extraction
Parse downloaded 10-K HTML filings and extract Item 1A (Risk Factors) 
and Item 7 (MD&A) sections for downstream NLP analysis.

In [3]:
import os
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

RAW_DIR = Path("../data/raw/sec-edgar-filings")
CLEANED_DIR = Path("../data/cleaned")
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

print("Ready.")

Ready.


In [4]:
def extract_sections(filepath):
    """Extract Item 1A and Item 7 text from a full-submission.txt 10-K filing."""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    
    # Strip HTML tags if present
    soup = BeautifulSoup(text, "lxml")
    text = soup.get_text(separator=" ")
    text = re.sub(r'\s+', ' ', text).strip()

    sections = {"item_1a": "", "item_7": ""}

    # Item 1A - Risk Factors
    match_1a = re.search(
        r'item\s+1a[\.\s]*risk factors(.+?)item\s+1b',
        text, re.IGNORECASE | re.DOTALL
    )
    if match_1a:
        sections["item_1a"] = match_1a.group(1).strip()[:50000]

    # Item 7 - MD&A
    match_7 = re.search(
        r'item\s+7[\.\s]*management.{0,50}discussion(.+?)item\s+7a',
        text, re.IGNORECASE | re.DOTALL
    )
    if match_7:
        sections["item_7"] = match_7.group(1).strip()[:50000]

    return sections

In [5]:
records = []

companies = [d for d in RAW_DIR.iterdir() if d.is_dir()]
print(f"Processing {len(companies)} companies...\n")

for company_dir in sorted(companies):
    ticker = company_dir.name
    filing_dir = company_dir / "10-K"
    
    if not filing_dir.exists():
        continue

    for filing in sorted(filing_dir.iterdir()):
        filepath = filing / "full-submission.txt"
        
        if not filepath.exists():
            continue
        
        try:
            sections = extract_sections(filepath)
            
            year_match = re.search(r'(\d{4})', filing.name)
            year = int(year_match.group(1)) if year_match else None
            
            records.append({
                "ticker": ticker,
                "year": year,
                "item_1a": sections["item_1a"],
                "item_7": sections["item_7"],
                "item_1a_words": len(sections["item_1a"].split()),
                "item_7_words": len(sections["item_7"].split()),
            })
            print(f"✓ {ticker} {year}")
        except Exception as e:
            print(f"✗ {ticker} {filing.name}: {e}")

df = pd.DataFrame(records)
print(f"\nExtracted {len(df)} filings")
print(f"Item 1A extracted: {(df['item_1a_words'] > 0).sum()}")
print(f"Item 7 extracted:  {(df['item_7_words'] > 0).sum()}")

Processing 51 companies...

✓ AAPL 0
✓ AAPL 0
✓ AAPL 0
✓ AAPL 0
✓ AAPL 0
✓ AAPL 0
✓ AAPL 0
✓ AAPL 1
✓ AAPL 1
✓ ABBV 1
✓ ABBV 1
✓ ABBV 1
✓ ABBV 1
✓ ABBV 1
✓ ABBV 1
✓ ABBV 1
✓ ABBV 1
✓ ABBV 1
✓ ABT 1
✓ ABT 1
✓ ABT 1
✓ ABT 1
✓ ABT 1
✓ ABT 1
✓ ABT 1
✓ ABT 1
✓ ABT 1
✓ ADBE 0
✓ ADBE 0
✓ ADBE 0
✓ ADBE 0
✓ ADBE 0
✓ ADBE 0
✓ ADBE 0
✓ ADBE 0
✓ ADBE 0
✓ AMD 0
✓ AMD 0
✓ AMD 0
✓ AMD 0
✓ AMD 0
✓ AMD 0
✓ AMD 0
✓ AMD 1
✓ AMD 1
✓ AMZN 1
✓ AMZN 1
✓ AMZN 1
✓ AMZN 1
✓ AMZN 1
✓ AMZN 1
✓ AMZN 1
✓ AMZN 1
✓ AMZN 1
✓ APD 0
✓ APD 0
✓ APD 0
✓ APD 0
✓ APD 0
✓ APD 0
✓ APD 0
✓ APD 1
✓ APD 1
✓ AVGO 1
✓ AVGO 1
✓ AVGO 1
✓ AVGO 1
✓ AVGO 1
✓ AVGO 1
✓ BAC 0
✓ BAC 0
✓ BAC 0
✓ BAC 0
✓ BAC 0
✓ BAC 0
✓ BAC 0
✓ BAC 0
✓ BAC 0
✓ BRK-B 0
✓ BRK-B 1
✓ BRK-B 1
✓ BRK-B 1
✓ BRK-B 1
✓ BRK-B 1
✓ BRK-B 1
✓ BRK-B 1
✓ BRK-B 1
✓ CAT 0
✓ CAT 0
✓ CAT 0
✓ CAT 0
✓ CAT 0
✓ CAT 0
✓ CAT 0
✓ CAT 0
✓ CAT 0
✓ COP 1
✓ COP 1
✓ COP 1
✓ COP 1
✓ COP 1
✓ COP 1
✓ COP 1
✓ COP 1
✓ COP 1
✓ COST 0
✓ COST 0
✓ COST 0
✓ COST 0
✓ COST 0
✓ COST 0
✓ COST 0
✓ COST 0


In [6]:
# Save cleaned data
output_path = CLEANED_DIR / "extracted_sections.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")
print()

# Preview
print(df[["ticker", "year", "item_1a_words", "item_7_words"]].sort_values(
    ["ticker", "year"]).head(20).to_string(index=False))

Saved to ../data/cleaned/extracted_sections.csv

ticker  year  item_1a_words  item_7_words
  AAPL     0              1            10
  AAPL     0              1            10
  AAPL     0              1            10
  AAPL     0              1            10
  AAPL     0              1            10
  AAPL     0              1            10
  AAPL     0              1            10
  AAPL     1              1            10
  AAPL     1              1            10
  ABBV     1           7236          8124
  ABBV     1           6964          8190
  ABBV     1              1            10
  ABBV     1              1            10
  ABBV     1              1            10
  ABBV     1              1            10
  ABBV     1              1            10
  ABBV     1              1            10
  ABBV     1              1            10
   ABT     1           3659          7536
   ABT     1           3813          7632
